In [1]:
# Core
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import MSELoss
# Data handling
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader,TensorDataset

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics & utilities
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Progress & debugging
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
import glob
import joblib

device = "mps" if torch.backends.mps.is_available() else "cpu"


In [2]:
all_files = sorted(glob.glob("data/monthly_hourly_load_values_*.xlsx"))
df_list = [pd.read_excel(file) for file in all_files]
df = pd.concat(df_list, ignore_index=True)
df['DateUTC'] = pd.to_datetime(df['DateUTC'])
df.set_index('DateUTC', inplace=True)

# Training: 2019–2024, Testing: 2025
train_df_full = df[df.index.year < 2025]
test_df = df[df.index.year == 2025]

# Split validation from end of 2024 (10% of training)
val_ratio = 0.1
val_size = int(len(train_df_full) * val_ratio)
val_df = train_df_full.iloc[-val_size:]
train_df = train_df_full.iloc[:-val_size]

def create_sequences(data, seq_len=24, output_len=1):
    X, y = [], []
    for i in range(seq_len, len(data) - output_len + 1):
        X.append(data[i - seq_len:i])
        y.append(data[i:i + output_len].flatten())
    return np.array(X), np.array(y)

scaler = MinMaxScaler()
scaler.fit(df[['Value']])
joblib.dump(scaler, 'scaler_minmax.pkl')
train_df['Value_normalized'] = scaler.transform(train_df[['Value']])
val_df['Value_normalized']   = scaler.transform(val_df[['Value']])
test_df['Value_normalized'] = scaler.transform(test_df[['Value']])


def prepare_multi_country_data_per_country_scaler(df, sequence_length=24, prediction_length=1, dataset_name="train/val"):
    """
    For each country, fit a StandardScaler, save it, and normalize values.
    """
    print(f"\n{'='*50}")
    print(f"PREPARING {dataset_name.upper()} WITH PER-COUNTRY SCALER")
    print(f"{'='*50}")

    df_clean = df.dropna(subset=['Value']).copy()
    df_clean = df_clean.sort_values(['CountryCode', 'DateUTC']).reset_index(drop=False)

    sequences = []
    targets = []
    countries = []
    timestamps = []

    for country in df_clean['CountryCode'].unique():
        country_data = df_clean[df_clean['CountryCode'] == country]
        scaler = StandardScaler()
        scaler.fit(country_data[['Value']])
        # Save scaler for later use
        scaler_filename = f"scaler_{country}.pkl"
        joblib.dump(scaler, scaler_filename)
        # Normalize values
        country_data['Value_normalized'] = scaler.transform(country_data[['Value']])
        values = country_data['Value_normalized'].values
        dates = country_data['DateUTC'].values
        X_seq, y_seq = create_sequences(values, seq_len=sequence_length, output_len=prediction_length)
        sequences.append(X_seq)
        targets.append(y_seq)
        countries.extend([country] * len(X_seq))
        timestamps.extend(dates[sequence_length:sequence_length+len(X_seq)])

    X = np.concatenate(sequences, axis=0) if sequences else np.array([])
    y = np.concatenate(targets, axis=0) if targets else np.array([])
    countries = np.array(countries)
    timestamps = np.array(timestamps)

    print(f"\nFinal dataset stats:")
    print(f"Total sequences: {len(X):,}")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Normalized X range: [{X.min():.3f}, {X.max():.3f}]")
    print(f"Normalized y range: [{y.min():.3f}, {y.max():.3f}]")

    unique_countries, counts = np.unique(countries, return_counts=True)
    print(f"\nCountry distribution in sequences:")
    for country, count in zip(unique_countries, counts):
        print(f"  {country}: {count:,} sequences ({count/len(X)*100:.1f}%)")

    return X, y

X_train, y_train = prepare_multi_country_data_per_country_scaler(
    train_df, sequence_length=24, prediction_length=1
)

X_val, y_val= prepare_multi_country_data_per_country_scaler(
    val_df, sequence_length=24, prediction_length=1
)

X_test, y_test = prepare_multi_country_data_per_country_scaler(
    test_df, sequence_length=24, prediction_length=1,dataset_name="test"
)

print(f"All datasets normalized with the same scaler")
print(f"Train range: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Val range:   [{X_val.min():.3f}, {X_val.max():.3f}]")
print(f"Test range:  [{X_test.min():.3f}, {X_test.max():.3f}]")



PREPARING TRAIN/VAL WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 1,649,186
X shape: (1649186, 24)
y shape: (1649186, 1)
Normalized X range: [-6.300, 43.175]
Normalized y range: [-6.300, 43.175]

Country distribution in sequences:
  AL: 24,912 sequences (1.5%)
  AT: 52,584 sequences (3.2%)
  BA: 47,982 sequences (2.9%)
  BE: 52,584 sequences (3.2%)
  BG: 52,584 sequences (3.2%)
  CH: 52,583 sequences (3.2%)
  CY: 27,617 sequences (1.7%)
  CZ: 52,578 sequences (3.2%)
  DE: 52,584 sequences (3.2%)
  DK: 52,583 sequences (3.2%)
  EE: 52,577 sequences (3.2%)
  ES: 52,583 sequences (3.2%)
  FI: 52,584 sequences (3.2%)
  FR: 52,519 sequences (3.2%)
  GB: 36,816 sequences (2.2%)
  GE: 25,806 sequences (1.6%)
  GR: 43,911 sequences (2.7%)
  HR: 43,800 sequences (2.7%)
  HU: 43,798 sequences (2.7%)
  IE: 38,057 sequences (2.3%)
  IT: 43,800 sequences (2.7%)
  LT: 43,798 sequences (2.7%)
  LU: 43,800 sequences (2.7%)
  LV: 43,799 sequences (2.7%)
  MD: 34,307 sequences (2.1%)
 

In [3]:
def evaluate_model(y_true, y_pred, scaler=None):
    # Remove any extra dimensions but keep the prediction_length dimension
    y_true = np.squeeze(y_true)
    y_pred = np.squeeze(y_pred)
    
    # Store original shapes for debugging
    original_shape_true = y_true.shape
    original_shape_pred = y_pred.shape
    
    print(f"Debug - y_true shape: {original_shape_true}, y_pred shape: {original_shape_pred}")
    
    if scaler:
        # For multi-step forecasting, we need to handle each time step separately
        # or flatten, transform, then reshape back
        if len(y_true.shape) > 1:
            # Multi-step forecasting - flatten, transform, then reshape back
            true_flat = y_true.reshape(-1, 1)
            pred_flat = y_pred.reshape(-1, 1)
            
            y_true = scaler.inverse_transform(true_flat).reshape(y_true.shape)
            y_pred = scaler.inverse_transform(pred_flat).reshape(y_pred.shape)
        else:
            # Single-step forecasting
            y_true = scaler.inverse_transform(y_true.reshape(-1, 1)).flatten()
            y_pred = scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    # Handle MAPE carefully to avoid division by zero
    # Use np.where to avoid division by very small numbers
    with np.errstate(divide='ignore', invalid='ignore'):
        ape = np.abs((y_true - y_pred) / np.where(np.abs(y_true) < 1e-8, 1e-8, y_true))
        mape = np.mean(ape) * 100
    
    print(f"Debug - After inverse - y_true range: [{y_true.min():.2f}, {y_true.max():.2f}]")
    print(f"Debug - After inverse - y_pred range: [{y_pred.min():.2f}, {y_pred.max():.2f}]")
    
    return round(rmse, 2), round(mae, 2), round(mape, 2), y_true, y_pred

def test_model(model, X_test, y_test, batch_size=32, scaler=None):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model.to(device)
    model.eval()
    
    if isinstance(X_test, np.ndarray):
        X_test = torch.FloatTensor(X_test)
    if isinstance(y_test, np.ndarray):
        y_test = torch.FloatTensor(y_test)
    
    test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for data, target in tqdm(test_loader, desc='Testing'):
            data, target = data.to(device), target.to(device)
            if len(data.shape) == 2:
                data = data.unsqueeze(-1)
            
            output = model(data)
            all_preds.append(output.cpu().numpy())
            all_targets.append(target.cpu().numpy())
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    
    # Debug: Check shapes before evaluation
    print(f"Before evaluation - all_targets shape: {all_targets.shape}")
    print(f"Before evaluation - all_preds shape: {all_preds.shape}")
    print(f"Before evaluation - all_targets range: [{all_targets.min():.3f}, {all_targets.max():.3f}]")
    print(f"Before evaluation - all_preds range: [{all_preds.min():.3f}, {all_preds.max():.3f}]")
    
    rmse, mae, mape, y_true_inv, y_pred_inv = evaluate_model(all_targets, all_preds, scaler=scaler)
    
    print(f"Test RMSE: {rmse}, MAE: {mae}, MAPE: {mape}%")
    
    return {
        'rmse': rmse,
        'mae': mae,
        'mape': mape,
        'y_true': y_true_inv,
        'y_pred': y_pred_inv
    }

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DualBranchTFT(nn.Module):
    def __init__(self, sequence_length, prediction_length, hidden_size=64, num_layers=2, 
                 dropout=0.1, conv_kernels=[3, 7, 15], num_attention_heads=4):
        super().__init__()
        self.sequence_length = sequence_length
        self.prediction_length = prediction_length
        self.hidden_size = hidden_size
        
        self.trend_branch = TrendExtractionBranch(
            input_size=1,
            hidden_size=hidden_size,
            conv_kernels=conv_kernels,
            dropout=dropout
        )
        
        # Encoder LSTM
        self.encoder_lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        
        # Attention mechanism
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_attention_heads,
            dropout=dropout,
            batch_first=True
        )
        
        self.fusion_type = "gated"  # Options: "concat", "gated", "weighted"
        
        if self.fusion_type == "concat":
            self.fusion_output_size = hidden_size * 2
            self.fusion_projection = nn.Linear(hidden_size * 2, hidden_size)
        elif self.fusion_type == "gated":
            self.fusion_gate = nn.Sequential(
                nn.Linear(hidden_size * 2, hidden_size),
                nn.Sigmoid()
            )
            self.fusion_output_size = hidden_size
        elif self.fusion_type == "weighted":
            self.fusion_weights = nn.Parameter(torch.ones(2))  # Two branches
            self.fusion_output_size = hidden_size
        
        # Layer normalizations
        self.lstm_norm = nn.LayerNorm(hidden_size)
        self.fusion_norm = nn.LayerNorm(self.fusion_output_size)
        self.output_layer = nn.Sequential(
            nn.Linear(self.fusion_output_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, prediction_length)
        )

    def forward(self, x):
        # x shape: (batch_size, sequence_length, 1)
        batch_size = x.size(0)
        
        trend_features = self.trend_branch(x)  # (batch_size, hidden_size)
        
        # LSTM encoding
        lstm_output, (hidden, cell) = self.encoder_lstm(x)
        lstm_output = self.lstm_norm(lstm_output)
        
        # Self-attention on LSTM outputs
        attended_output, attention_weights = self.attention(
            lstm_output, lstm_output, lstm_output
        )
        
        # Get the final temporal representation (last time step)
        temporal_features = attended_output[:, -1, :]  # (batch_size, hidden_size)
        
        if self.fusion_type == "concat":
            # Simple concatenation with projection
            fused_features = torch.cat([trend_features, temporal_features], dim=-1)
            fused_features = self.fusion_projection(fused_features)
            
        elif self.fusion_type == "gated":
            # Gated fusion - learn which branch to trust more
            combined = torch.cat([trend_features, temporal_features], dim=-1)
            gate = self.fusion_gate(combined)
            fused_features = gate * trend_features + (1 - gate) * temporal_features
            
        elif self.fusion_type == "weighted":
            # Weighted sum of branches
            weights = F.softmax(self.fusion_weights, dim=0)
            fused_features = weights[0] * trend_features + weights[1] * temporal_features
        
        fused_features = self.fusion_norm(fused_features)
        
        output = self.output_layer(fused_features)  # (batch_size, prediction_length)
        
        return output


class TrendExtractionBranch(nn.Module):
    """CNN-based branch for multi-scale trend extraction"""
    
    def __init__(self, input_size, hidden_size, conv_kernels=[3, 7, 15], dropout=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.conv_kernels = conv_kernels
        self.num_kernels = len(conv_kernels)
    
        # Use divisible allocation
        base_channels = hidden_size // self.num_kernels
        remainder = hidden_size % self.num_kernels
        self.output_channels = [base_channels + (1 if i < remainder else 0) 
                              for i in range(self.num_kernels)]
        
        # Parallel convolutional layers with different kernel sizes
        self.conv_layers = nn.ModuleList()
        for i, kernel_size in enumerate(conv_kernels):
            padding = (kernel_size - 1) // 2  # Same padding
            conv_block = nn.Sequential(
                nn.Conv1d(
                    in_channels=input_size,
                    out_channels=self.output_channels[i],
                    kernel_size=kernel_size,
                    padding=padding,
                    padding_mode='replicate'
                ),
                nn.BatchNorm1d(self.output_channels[i]),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
            self.conv_layers.append(conv_block)
        
        # Global pooling to get trend representation
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Feature projection to ensure correct hidden_size
        self.feature_projection = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.LayerNorm(hidden_size)
        )
        
    def forward(self, x):
        # x shape: (batch_size, sequence_length, 1)
        # Rearrange for conv1d: (batch_size, channels, sequence_length)
        x_conv = x.transpose(1, 2)  # (batch_size, 1, sequence_length)
        
        # Apply parallel convolutions
        conv_outputs = []
        for conv_layer in self.conv_layers:
            conv_out = conv_layer(x_conv)  # (batch_size, output_channel_i, seq_len)
            conv_outputs.append(conv_out)
        
        # Concatenate along channel dimension
        combined_conv = torch.cat(conv_outputs, dim=1)  # (batch_size, hidden_size, seq_len)
        
        # Global average pooling to get trend representation
        trend_rep = self.global_pool(combined_conv)  # (batch_size, hidden_size, 1)
        trend_rep = trend_rep.squeeze(-1)  # (batch_size, hidden_size)
        
        # Final projection to ensure correct dimensions
        trend_features = self.feature_projection(trend_rep)  # (batch_size, hidden_size)
        
        return trend_features


sequence_length = 24
prediction_length = 1
hidden_size =128

# # # Initialize the dual-branch model
# model_full = DualBranchTFT(
#     sequence_length=sequence_length,
#     prediction_length=prediction_length,
#     hidden_size=hidden_size,
#     num_layers=2,
#     dropout=0.1,
#     conv_kernels=[3, 7, 15]  # Multi-scale trend extraction
# )
# model.load_state_dict(torch.load("model_state_dict.pth"))

In [5]:
class FTE(nn.Module):
    """Ablation: No Trend Extraction"""
    def __init__(self, sequence_length, prediction_length,
                 hidden_size=64, num_layers=2, dropout=0.1, num_attention_heads=4):
        super().__init__()

        self.encoder_lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_attention_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(hidden_size)

        self.output_layer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, prediction_length)
        )

    def forward(self, x):
        lstm_out, _ = self.encoder_lstm(x)
        lstm_out = self.norm(lstm_out)

        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        temporal_features = attn_out[:, -1, :]

        return self.output_layer(temporal_features)


In [6]:
class FTE_NoAttention(nn.Module):
    """Ablation: No Trend Extraction, No Attention"""
    def __init__(self, sequence_length, prediction_length,
                 hidden_size=64, num_layers=2, dropout=0.1):
        super().__init__()

        self.encoder_lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.norm = nn.LayerNorm(hidden_size)

        self.output_layer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, prediction_length)
        )

    def forward(self, x):
        lstm_out, _ = self.encoder_lstm(x)
        temporal_features = self.norm(lstm_out[:, -1, :])
        return self.output_layer(temporal_features)


In [7]:
class DualBranch_NoGate(nn.Module):
    """Ablation: No Gated Fusion (Concat Fusion)"""
    def __init__(self, sequence_length, prediction_length,
                 hidden_size=64, num_layers=2, dropout=0.1,
                 conv_kernels=[3, 7, 15], num_attention_heads=4):
        super().__init__()

        self.trend_branch = TrendExtractionBranch(
            input_size=1,
            hidden_size=hidden_size,
            conv_kernels=conv_kernels,
            dropout=dropout
        )

        self.encoder_lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_attention_heads,
            dropout=dropout,
            batch_first=True
        )

        self.fusion_projection = nn.Linear(hidden_size * 2, hidden_size)
        self.norm = nn.LayerNorm(hidden_size)

        self.output_layer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, prediction_length)
        )

    def forward(self, x):
        trend_feat = self.trend_branch(x)

        lstm_out, _ = self.encoder_lstm(x)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        temporal_feat = attn_out[:, -1, :]

        fused = torch.cat([trend_feat, temporal_feat], dim=-1)
        fused = self.norm(self.fusion_projection(fused))

        return self.output_layer(fused)


In [8]:
def test_model_per_country(model, df, sequence_length=24, prediction_length=1, batch_size=32):
    """
    Test the model for each country using its own saved StandardScaler.
    """
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model.to(device)
    model.eval()

    countries = df['CountryCode'].unique()
    results_dict = {}

    for country in countries:
        country_df = df[df['CountryCode'] == country].copy()
        scaler_path = f"scaler_{country}.pkl"
        try:
            scaler = joblib.load(scaler_path)
        except FileNotFoundError:
            print(f"Scaler for {country} not found. Skipping.")
            continue

        # Prepare sequences for this country
        X_test, y_test = prepare_multi_country_data_per_country_scaler(
            country_df, sequence_length=sequence_length, prediction_length=prediction_length, dataset_name="test"
        )
        if len(X_test) == 0:
            print(f"Skipping {country}: Not enough data for sequence_length={sequence_length}")
            continue

        # Convert to tensors
        if isinstance(X_test, np.ndarray):
            X_test = torch.FloatTensor(X_test)
        if isinstance(y_test, np.ndarray):
            y_test = torch.FloatTensor(y_test)

        test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        all_preds = []
        all_targets = []

        with torch.no_grad():
            for data, target in tqdm(test_loader, desc=f'Testing {country}'):
                data, target = data.to(device), target.to(device)
                if len(data.shape) == 2:
                    data = data.unsqueeze(-1)
                output = model(data)
                all_preds.append(output.cpu().numpy())
                all_targets.append(target.cpu().numpy())

        all_preds = np.concatenate(all_preds, axis=0)
        all_targets = np.concatenate(all_targets, axis=0)

        # Inverse transform using country scaler
        rmse, mae, mape, y_true_inv, y_pred_inv = evaluate_model(all_targets, all_preds, scaler=scaler)
        print(f"{country} - RMSE: {rmse}, MAE: {mae}, MAPE: {mape}%")

        results_dict[country] = {
            'rmse': rmse,
            'mae': mae,
            'mape': mape,
            'y_true': y_true_inv,
            'y_pred': y_pred_inv
        }

    return results_dict

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


def train_model(model, X_train, y_train, X_val, y_val, epochs=50, batch_size=32, learning_rate=1e-4):
    """
    Simple PyTorch training function with tqdm progress bars
    """
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model.to(device)
    
    # Convert data to tensors
    if isinstance(X_train, np.ndarray):
        X_train = torch.FloatTensor(X_train)
    if isinstance(y_train, np.ndarray):
        y_train = torch.FloatTensor(y_train)
    if isinstance(X_val, np.ndarray):
        X_val = torch.FloatTensor(X_val)
    if isinstance(y_val, np.ndarray):
        y_val = torch.FloatTensor(y_val)
    
    # Create DataLoaders
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    val_dataset = torch.utils.data.TensorDataset(X_val, y_val)
    
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Optimizer and loss
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': []
    }
    
    print("Starting training...")
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        # Create progress bar for training
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        
        for batch_idx, (data, target) in enumerate(train_pbar):
            data, target = data.to(device), target.to(device)
            
            # Add channel dimension if needed (batch_size, seq_len) -> (batch_size, seq_len, 1)
            if len(data.shape) == 2:
                data = data.unsqueeze(-1)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            # Update progress bar
            train_pbar.set_postfix({
                'Loss': f'{loss.item():.6f}',
                'Avg Loss': f'{train_loss/(batch_idx+1):.6f}'
            })
        
        avg_train_loss = train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
        
        with torch.no_grad():
            for data, target in val_pbar:
                data, target = data.to(device), target.to(device)
                if len(data.shape) == 2:
                    data = data.unsqueeze(-1)
                
                output = model(data)
                loss = criterion(output, target)
                val_loss += loss.item()
                
                # Update validation progress bar
                val_pbar.set_postfix({
                    'Loss': f'{loss.item():.6f}',
                    'Avg Loss': f'{val_loss/(len(val_pbar)+1):.6f}'
                })
        
        avg_val_loss = val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)
        
        # Print epoch summary
        print(f'Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
    
    print("Training completed!")
    return model,history

In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
sequence_length = 24
prediction_length = 1
hidden_size =128
scaler = joblib.load('scaler_minmax.pkl')
model_no_trend = FTE(
    sequence_length=sequence_length,
    prediction_length=prediction_length,
    hidden_size=hidden_size,
    num_layers=2,
    dropout=0.1
)
model_no_attn = FTE_NoAttention(
    sequence_length=sequence_length,
    prediction_length=prediction_length,
    hidden_size=hidden_size,
    num_layers=2,
    dropout=0.1
)
model_no_gate = DualBranch_NoGate(
    sequence_length=sequence_length,
    prediction_length=prediction_length,
    hidden_size=hidden_size,
    num_layers=2,
    dropout=0.1,
    conv_kernels=[3, 7, 15]
)

model_no_trend,history_no_trend = train_model(
    model_no_trend, X_train, y_train, X_val, y_val,
    epochs=3, batch_size=32, learning_rate=1e-4
)
model_no_attn,history_no_attn = train_model(
    model_no_attn, X_train, y_train, X_val, y_val,
    epochs=3, batch_size=32, learning_rate=1e-4
)
model_no_gate,history_no_gate = train_model(
    model_no_gate, X_train, y_train, X_val, y_val,
    epochs=3, batch_size=32, learning_rate=1e-4
)

results_no_trend = test_model_per_country(
    model_no_trend, test_df, sequence_length=24, prediction_length=1
)
results_no_attn = test_model_per_country(
    model_no_attn, test_df, sequence_length=24, prediction_length=1
)
results_no_gate = test_model_per_country(
    model_no_gate, test_df, sequence_length=24, prediction_length=1
)

Using device: mps
Starting training...


Epoch 1/3 [Val]: 100%|██████████| 5714/5714 [00:25<00:00, 225.16it/s, Loss=2.381632, Avg Loss=0.021485]


Epoch 1/3 - Train Loss: 0.037564, Val Loss: 0.021489


Epoch 2/3 [Val]: 100%|██████████| 5714/5714 [00:25<00:00, 220.81it/s, Loss=2.369025, Avg Loss=0.020895]


Epoch 2/3 - Train Loss: 0.030524, Val Loss: 0.020898


...

Debug - y_true shape: (4301,), y_pred shape: (4301,)
Debug - After inverse - y_true range: [496.43, 2248.73]
Debug - After inverse - y_pred range: [485.35, 2211.50]
SI - RMSE: 52.6, MAE: 38.45, MAPE: 2.809999942779541%

PREPARING TEST WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 4,318
X shape: (4318, 24)
y shape: (4318, 1)
Normalized X range: [-2.258, 2.545]
Normalized y range: [-2.258, 2.545]

Country distribution in sequences:
  SK: 4,318 sequences (100.0%)


Testing SK: 100%|██████████| 135/135 [00:00<00:00, 210.57it/s]


Debug - y_true shape: (4318,), y_pred shape: (4318,)
Debug - After inverse - y_true range: [1952.00, 4157.00]
Debug - After inverse - y_pred range: [1973.58, 4177.90]
SK - RMSE: 66.71, MAE: 48.18, MAPE: 1.6100000143051147%

PREPARING TEST WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 4,320
X shape: (4320, 24)
y shape: (4320, 1)
Normalized X range: [-1.846, 2.403]
Normalized y range: [-1.846, 2.403]

Country distribution in sequences:
  XK: 4,320 sequences (100.0%)


Testing XK: 100%|██████████| 135/135 [00:00<00:00, 211.16it/s]

Debug - y_true shape: (4320,), y_pred shape: (4320,)
Debug - After inverse - y_true range: [311.43, 1418.44]
Debug - After inverse - y_pred range: [325.32, 1390.54]
XK - RMSE: 26.42, MAE: 19.43, MAPE: 2.7100000381469727%


In [11]:
# Calculate average metrics excluding countries with MAPE > 10%
def print_average_metrics_exclude_high_mape(results_dict, mape_threshold=10):
    filtered = [v for v in results_dict.values() if v['mape'] <= mape_threshold]
    if not filtered:
        print("No countries with MAPE below threshold.")
        return
    avg_rmse = np.mean([v['rmse'] for v in filtered])
    avg_mae = np.mean([v['mae'] for v in filtered])
    avg_mape = np.mean([v['mape'] for v in filtered])
    print(f"Average metrics for countries with MAPE ≤ {mape_threshold}%:")
    print(f"  RMSE: {avg_rmse:.2f}")
    print(f"  MAE:  {avg_mae:.2f}")
    print(f"  MAPE: {avg_mape:.2f}%")

# Example usage:
print_average_metrics_exclude_high_mape(results_no_trend, mape_threshold=15)
print_average_metrics_exclude_high_mape(results_no_attn, mape_threshold=15)
print_average_metrics_exclude_high_mape(results_no_gate, mape_threshold=15)

Average metrics for countries with MAPE ≤ 15%:
  RMSE: 196.96
  MAE:  139.35
  MAPE: 1.97%
Average metrics for countries with MAPE ≤ 15%:
  RMSE: 199.27
  MAE:  140.48
  MAPE: 1.98%
Average metrics for countries with MAPE ≤ 15%:
  RMSE: 199.69
  MAE:  141.80
  MAPE: 2.01%


In [12]:
import torch
import torch.nn as nn

class DLinear(nn.Module):
    """
    DLinear: Decomposition-Linear Model for Time Series Forecasting
    """
    def __init__(self, seq_len, pred_len, enc_in):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.enc_in = enc_in

        # Trend & seasonal linear layers
        self.linear_seasonal = nn.Linear(seq_len, pred_len)
        self.linear_trend = nn.Linear(seq_len, pred_len)

    def moving_avg(self, x, kernel_size=25):
        padding = (kernel_size - 1) // 2
        x_padded = torch.nn.functional.pad(x, (padding, padding), mode='replicate')
        return torch.nn.functional.avg_pool1d(x_padded, kernel_size, stride=1)

    def series_decompose(self, x):
        # x: (B, T, C) → (B, C, T)
        x = x.permute(0, 2, 1)
        trend = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend

    def forward(self, x):
        # x: (batch, seq_len, channels)
        seasonal, trend = self.series_decompose(x)

        seasonal = self.linear_seasonal(seasonal)
        trend = self.linear_trend(trend)

        out = seasonal + trend
        return out.permute(0, 2, 1)  # (B, pred_len, C)


In [13]:
model_dlinear = DLinear(
    seq_len=sequence_length,
    pred_len=prediction_length,
    enc_in=1
)
model_dlinear,history_dlinear = train_model(
    model_dlinear, X_train, y_train, X_val, y_val,
    epochs=3, batch_size=32, learning_rate=1e-4
)
results_dlinear = test_model_per_country(
    model_dlinear, test_df, sequence_length=24, prediction_length=1
)
print_average_metrics_exclude_high_mape(results_dlinear, mape_threshold=15)

Using device: mps
Starting training...


Epoch 1/3 [Val]: 100%|██████████| 5714/5714 [00:09<00:00, 630.55it/s, Loss=4.275641, Avg Loss=0.971388]


Epoch 1/3 - Train Loss: 1.002364, Val Loss: 0.971558


Epoch 2/3 [Val]: 100%|██████████| 5714/5714 [00:12<00:00, 440.85it/s, Loss=4.378664, Avg Loss=0.980497]


Epoch 2/3 - Train Loss: 0.999259, Val Loss: 0.980669


...

Debug - y_true shape: (4301,), y_pred shape: (4301,)
Debug - After inverse - y_true range: [496.43, 2248.73]
Debug - After inverse - y_pred range: [1387.05, 1463.54]
SI - RMSE: 297.55, MAE: 240.15, MAPE: 18.34000015258789%

PREPARING TEST WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 4,318
X shape: (4318, 24)
y shape: (4318, 1)
Normalized X range: [-2.258, 2.545]
Normalized y range: [-2.258, 2.545]

Country distribution in sequences:
  SK: 4,318 sequences (100.0%)


Testing SK: 100%|██████████| 135/135 [00:00<00:00, 1231.81it/s]


Debug - y_true shape: (4318,), y_pred shape: (4318,)
Debug - After inverse - y_true range: [1952.00, 4157.00]
Debug - After inverse - y_pred range: [2942.09, 3041.78]
SK - RMSE: 441.0, MAE: 357.78, MAPE: 12.319999694824219%

PREPARING TEST WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 4,320
X shape: (4320, 24)
y shape: (4320, 1)
Normalized X range: [-1.846, 2.403]
Normalized y range: [-1.846, 2.403]

Country distribution in sequences:
  XK: 4,320 sequences (100.0%)


Testing XK: 100%|██████████| 135/135 [00:00<00:00, 1096.14it/s]

Debug - y_true shape: (4320,), y_pred shape: (4320,)
Debug - After inverse - y_true range: [311.43, 1418.44]
Debug - After inverse - y_pred range: [772.21, 820.08]
XK - RMSE: 247.85, MAE: 207.26, MAPE: 30.709999084472656%
Average metrics for countries with MAPE ≤ 15%:
  RMSE: 1928.75
  MAE:  1590.63
  MAPE: 13.24%
